# Olist Delivery Delay Prediction
### Brazilian E-Commerce Public Dataset | Capstone Project
#### Notebook 04 : Feature Engineering

---

| Section | Details |
|---------|----------|
| **Input** | `olist_merged.csv` (96,455 × 38), `items_enriched.csv` (112,650 × 22) |
| **Output** | `olist_features.csv` — model-ready dataset, `olist_features_cluster.csv` — clustering variant |
| **Previous** | `03_eda.ipynb` |
| **Next** | `05_modeling.ipynb`, `06_clustering.ipynb` |

**Goal:** Convert the merged order-level dataset into a fully engineered feature matrix.
Every decision here is driven by EDA findings from NB03.

---

## Engineering Plan (EDA-Driven)

| # | Feature Group | Source | EDA Finding |
|---|--------------|--------|-------------|
| 1 | **Seller features** | `items_enriched` | Seller late rate varies 0–40%+; geography is strongest signal |
| 2 | **Seller → customer distance** | geo coordinates | Northern states have 2–3× late rate of Southeast |
| 3 | **Product category flags** | `items_enriched` | Bulky categories (furniture, appliances) have highest late rates |
| 4 | **Temporal features** | `order_purchase_timestamp` | Monthly spikes, DOW variation, hour-of-day pattern observed |
| 5 | **Logistics timing** | timestamp deltas | Approval delay and carrier handoff window signal operational efficiency |
| 6 | **Delivery buffer** | `estimated_date – purchase_date` | The 'promise window' given to the customer |
| 7 | **Log transforms** | `total_price`, `freight`, `weight` | Right-skewed distributions confirmed in NB03 |
| 8 | **Leakage removal** | `delivery_days`, `review_score`, timestamps | Post-delivery — must be excluded before modeling |
| 9 | **Encoding** | `customer_state`, `payment_type` | Categorical → numeric for model compatibility |

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## Setup and Imports

In [4]:
import pandas as pd
import numpy as np
import warnings
from math import radians, sin, cos, sqrt, atan2

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

PATH = '/content/drive/MyDrive/Colab Notebooks/Olist/data/'

date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date', 'max_shipping_limit_date',
    'min_shipping_limit_date'
]

df    = pd.read_csv(PATH + 'olist_merged.csv',    parse_dates=date_cols)
items = pd.read_csv(PATH + 'items_enriched.csv',  parse_dates=['shipping_limit_date'])
geo   = pd.read_csv(PATH + 'zip_geo_centroids.csv')

print(f'olist_merged   : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'items_enriched : {items.shape[0]:,} rows × {items.shape[1]} columns')
print(f'geolocation    : {geo.shape[0]:,} rows × {geo.shape[1]} columns')
print(f'\nTarget rate (is_late=1) : {df["is_late"].mean()*100:.2f}%')

olist_merged   : 96,455 rows × 38 columns
items_enriched : 112,650 rows × 22 columns
geolocation    : 19,015 rows × 3 columns

Target rate (is_late=1) : 8.11%


---
## Step 1 — Seller Features

**EDA finding (NB03 §6b):** Seller late rates vary enormously (0% to 40%+).
Seller identity is a strong signal beyond just geography.

**Strategy:** We define the "bottleneck seller" as the seller with the latest
`shipping_limit_date` in each order — they are the last to ship and most likely
to cause a delay. All seller-level features attach to this seller.

We compute seller features using **leave-one-out / full history** approach:
seller historical late rate is computed from all orders in the dataset.
In production this would use only past orders — flagged for NB05.

### 1a — Identify the bottleneck seller per order

In [5]:
# ── Bottleneck seller = seller with latest shipping_limit_date per order ──────
bottleneck = (
    items
    .sort_values('shipping_limit_date', ascending=False)
    .drop_duplicates(subset='order_id', keep='first')
    [['order_id', 'seller_id', 'seller_state', 'seller_zip_code_prefix']]
    .rename(columns={
        'seller_id'               : 'bottleneck_seller_id',
        'seller_state'            : 'bottleneck_seller_state',
        'seller_zip_code_prefix'  : 'bottleneck_seller_zip'
    })
)

print(f'Bottleneck seller identified for {len(bottleneck):,} orders')
print(f'Unique bottleneck sellers : {bottleneck["bottleneck_seller_id"].nunique():,}')
print(f'\nSample:')
print(bottleneck.head(3).to_string(index=False))

Bottleneck seller identified for 98,666 orders
Unique bottleneck sellers : 3,089

Sample:
                        order_id             bottleneck_seller_id bottleneck_seller_state  bottleneck_seller_zip
c2bb89b5c1dd978d507284be78a04cb2 7a241947449cc45dbfda4f9d0798d9d0                      MG                  37590
13bdf405f961a6deec817d817f5c6624 7a241947449cc45dbfda4f9d0798d9d0                      MG                  37590
9c94a4ea2f7876660fa6f1b59b69c8e6 7a241947449cc45dbfda4f9d0798d9d0                      MG                  37590


### 1b — Seller historical late rate and order count

In [6]:
# ── Build seller performance table from items_enriched + target ──────────────
# Join target to items to get per-seller stats
items_with_target = items.merge(
    df[['order_id', 'is_late']], on='order_id', how='inner'
)

# Step 1: define the same 80/20 split used in NB05
split_idx = int(len(df) * 0.80)
train_order_ids = set(df['order_id'].iloc[:split_idx])

# Step 2: filter items_with_target to train orders only
items_train_only = items_with_target[
    items_with_target['order_id'].isin(train_order_ids)
]

# Step 3: compute seller stats on train only
seller_perf = (
    items_train_only             # ← was items_with_target (full data)
    .groupby('seller_id', as_index=False)
    .agg(
        seller_order_count = ('order_id', 'nunique'),
        seller_late_rate   = ('is_late', 'mean'),
    )
)

# rest stays exactly the same
k = 10
global_late_rate = df['is_late'].mean()
seller_perf['seller_late_rate_smooth'] = (
    (seller_perf['seller_order_count'] * seller_perf['seller_late_rate'] + k * global_late_rate)
    / (seller_perf['seller_order_count'] + k)
)

---
## Step 2 — Seller → Customer Distance

**EDA finding (NB03 §5):** Geography is the strongest signal.
Northern states show 2–3× the late rate of Southeastern states.

**Strategy:**
1. Aggregate geolocation to one centroid per zip code prefix
2. Attach seller centroid using `bottleneck_seller_zip`
3. Compute Haversine distance between seller and customer coordinates

In [7]:
# ── Aggregate geolocation → one centroid per zip prefix ──────────────────────
geo_lookup = (
    geo
    .groupby('zip_code_prefix', as_index=False)
    .agg(lat=('lat', 'mean'), lng=('lng', 'mean'))
)

seller_geo = geo_lookup.rename(columns={
    'zip_code_prefix': 'bottleneck_seller_zip',
    'lat': 'seller_lat',
    'lng': 'seller_lng'
})

print(f'Geo lookup: {len(geo_lookup):,} unique zip prefixes')

Geo lookup: 19,015 unique zip prefixes


In [8]:
# ── Haversine distance function ───────────────────────────────────────────────
def haversine_km(lat1, lng1, lat2, lng2):
    """Compute great-circle distance in km between two coordinate pairs."""
    R = 6371.0  # Earth radius in km
    rlat1, rlng1, rlat2, rlng2 = map(radians, [lat1, lng1, lat2, lng2])
    dlat = rlat2 - rlat1
    dlng = rlng2 - rlng1
    a = sin(dlat / 2)**2 + cos(rlat1) * cos(rlat2) * sin(dlng / 2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))


def compute_distance_vectorized(df_merged):
    """Vectorised Haversine using numpy for speed."""
    lat1 = np.radians(df_merged['seller_lat'].values)
    lng1 = np.radians(df_merged['seller_lng'].values)
    lat2 = np.radians(df_merged['customer_lat'].values)
    lng2 = np.radians(df_merged['customer_lng'].values)

    dlat = lat2 - lat1
    dlng = lng2 - lng1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2)**2
    return 6371.0 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


print('Distance functions defined.')

Distance functions defined.


In [9]:
# ── Attach seller coordinates and compute distance ────────────────────────────
# Step 1: merge bottleneck seller identity onto df
df = df.merge(bottleneck, on='order_id', how='left')

# Step 2: merge seller geo
df = df.merge(seller_geo, on='bottleneck_seller_zip', how='left')

# Step 3: compute Haversine distance where both coordinates available
has_coords = (
    df['seller_lat'].notna() & df['seller_lng'].notna() &
    df['customer_lat'].notna() & df['customer_lng'].notna()
)

df['seller_customer_distance_km'] = np.nan
df.loc[has_coords, 'seller_customer_distance_km'] = compute_distance_vectorized(
    df[has_coords]
)

# Fill missing distances with median (orders missing geo data)
median_dist = df['seller_customer_distance_km'].median()
n_missing   = df['seller_customer_distance_km'].isna().sum()
df['seller_customer_distance_km'] = df['seller_customer_distance_km'].fillna(median_dist)

print(f'Distance computed for {has_coords.sum():,} orders ({has_coords.mean()*100:.1f}%)')
print(f'Missing coordinates filled with median ({median_dist:.1f} km): {n_missing:,} orders')
print(f'\nDistance stats (km):')
print(df['seller_customer_distance_km'].describe(percentiles=[.25,.5,.75,.95]).to_string())

Distance computed for 95,978 orders (99.5%)
Missing coordinates filled with median (433.9 km): 477 orders

Distance stats (km):
count   96455.0000
mean      600.0240
std       592.4171
min         0.0000
25%       189.1319
50%       433.8829
75%       796.3973
95%      2093.8506
max      8677.9116


---
## Step 3 — Attach Seller Performance Features

In [10]:
# ── Join seller performance onto main df ──────────────────────────────────────
df = df.merge(
    seller_perf[['seller_id', 'seller_order_count', 'seller_late_rate',
                 'seller_late_rate_smooth']],
    left_on='bottleneck_seller_id',
    right_on='seller_id',
    how='left'
).drop(columns=['seller_id'])  # drop the join key duplicate

# Fill any remaining NaN (orders with no item records — very rare)
df['seller_late_rate']        = df['seller_late_rate'].fillna(global_late_rate)
df['seller_late_rate_smooth'] = df['seller_late_rate_smooth'].fillna(global_late_rate)
df['seller_order_count']      = df['seller_order_count'].fillna(0).astype(int)

print('Seller performance features attached.')
print(f"  seller_late_rate_smooth  — mean: {df['seller_late_rate_smooth'].mean():.4f}  "
      f"std: {df['seller_late_rate_smooth'].std():.4f}")
print(f"  seller_order_count       — median: {df['seller_order_count'].median():.0f}  "
      f"max: {df['seller_order_count'].max()}")

Seller performance features attached.
  seller_late_rate_smooth  — mean: 0.0789  std: 0.0418
  seller_order_count       — median: 121  max: 1466


---
## Step 4 — Product Category Flags

**EDA finding (NB03 §6):** Categories encode shipping difficulty.
Furniture, home_appliances, and similar bulky categories have the highest late rates.

**Strategy:** Two levels of category features:
- **Grouped flags** (`has_cat_group_*`): 6 broad groups — better for clustering, interpretable
- **Dominant category** (`dominant_category`): single most frequent category per order — for tree models

> We do NOT use full One-Hot Encoding of all 71 categories — too sparse.
> EDA showed the physical dimensions already capture most of the category signal numerically.

In [11]:
# ── Category group mapping (based on EDA late-rate profile) ──────────────────
# Groups ordered roughly by late-rate risk (bulky = highest risk)
CATEGORY_GROUPS = {
    # Bulky / heavy — highest late rates in NB03
    'bulky_home': [
        'furniture_decor', 'furniture_living_room', 'furniture_bedroom',
        'furniture_mattress_and_upholstery', 'home_appliances',
        'home_appliances_2', 'garden_tools', 'construction_tools_construction',
        'construction_tools_lights', 'construction_tools_safety',
        'fixed_telephony', 'air_conditioning',
    ],
    # Electronics — moderate-high
    'electronics': [
        'computers_accessories', 'computers', 'electronics', 'telephony',
        'audio', 'tablets_printing_image', 'consoles_games', 'pc_gamer',
        'signaling_and_security', 'small_appliances',
        'small_appliances_home_oven_and_coffee',
    ],
    # Fashion / accessories — lower late rate
    'fashion': [
        'fashion_bags_accessories', 'fashion_female_clothing',
        'fashion_male_clothing', 'fashion_shoes', 'fashion_underwear_beach',
        'fashion_childrens_clothes', 'watches_gifts', 'luggage_accessories',
    ],
    # Sports / leisure
    'sports_leisure': [
        'sports_leisure', 'toys', 'baby', 'diapers_and_hygiene',
        'musical_instruments',
    ],
    # Health / beauty / personal care
    'health_beauty': [
        'health_beauty', 'perfumery', 'housewares',
        'home_confort', 'home_comfort_2', 'bed_bath_table',
    ],
    # Books / media / food — lightest, lowest late rate
    'media_food': [
        'books_general_interest', 'books_imported', 'books_technical',
        'dvds_blu_ray', 'cds_dvds_musicals', 'music',
        'food', 'food_drink', 'drinks', 'la_cuisine',
        'flowers', 'christmas_supplies', 'party_supplies',
    ],
}

# Build reverse lookup: category_name → group
cat_to_group = {}
for group, cats in CATEGORY_GROUPS.items():
    for cat in cats:
        cat_to_group[cat] = group

print(f'Category groups defined: {len(CATEGORY_GROUPS)}')
print(f'Categories mapped: {len(cat_to_group)}')

# Assign group to each item
items['category_group'] = items['product_category_name_english'].map(cat_to_group).fillna('other')

# Verify coverage
covered = items['category_group'].ne('other').mean()
print(f'Coverage (non-other items): {covered*100:.1f}%')

Category groups defined: 6
Categories mapped: 55
Coverage (non-other items): 83.8%


In [12]:
# ── Build order-level category features ──────────────────────────────────────

# 1. Grouped flags — has_cat_group_* (binary per group)
cat_flags = (
    items
    .assign(val=1)
    .drop_duplicates(subset=['order_id', 'category_group'])
    .pivot_table(index='order_id', columns='category_group',
                 values='val', fill_value=0)
    .rename(columns=lambda c: f'has_cat_{c}')
    .reset_index()
)
# Flatten multi-index if needed
if isinstance(cat_flags.columns, pd.MultiIndex):
    cat_flags.columns = ['order_id'] + [c[1] for c in cat_flags.columns[1:]]

# 2. Dominant category (most frequent category per order)
dominant_cat = (
    items
    .groupby('order_id')['product_category_name_english']
    .agg(lambda x: x.mode()[0] if not x.mode().empty else 'unknown')
    .reset_index()
    .rename(columns={'product_category_name_english': 'dominant_category'})
)

# 3. Dominant category group
dominant_cat['dominant_category_group'] = dominant_cat['dominant_category'].map(cat_to_group).fillna('other')

# Merge onto df
df = df.merge(cat_flags, on='order_id', how='left')
df = df.merge(dominant_cat, on='order_id', how='left')

# Fill NaN (orders with no item records)
cat_flag_cols = [c for c in df.columns if c.startswith('has_cat_')]
df[cat_flag_cols] = df[cat_flag_cols].fillna(0).astype(int)
df['dominant_category']       = df['dominant_category'].fillna('unknown')
df['dominant_category_group'] = df['dominant_category_group'].fillna('other')

print('Category features created:')
for col in cat_flag_cols:
    print(f"  {col:<30s}: {df[col].sum():,} orders ({df[col].mean()*100:.1f}%)")
print(f"\n  dominant_category_group distribution:")
print(df['dominant_category_group'].value_counts().to_string())

Category features created:
  has_cat_bulky_home            : 12,816 orders (13.3%)
  has_cat_electronics           : 15,568 orders (16.1%)
  has_cat_fashion               : 8,795 orders (9.1%)
  has_cat_health_beauty         : 27,074 orders (28.1%)
  has_cat_media_food            : 2,060 orders (2.1%)
  has_cat_other                 : 15,991 orders (16.6%)
  has_cat_sports_leisure        : 14,749 orders (15.3%)

  dominant_category_group distribution:
dominant_category_group
health_beauty     26970
other             15815
electronics       15532
sports_leisure    14657
bulky_home        12695
fashion            8741
media_food         2045


---
## Step 5 — Temporal Features

**EDA findings (NB03 §4):**
- Monthly spikes visible (Black Friday, holiday season)
- Day-of-week variation is modest but present
- Purchase hour shows clear pattern: late-night orders have higher late rates

In [13]:
# ── Temporal features from order_purchase_timestamp ──────────────────────────
ts = df['order_purchase_timestamp']

df['purchase_year']       = ts.dt.year
df['purchase_month']      = ts.dt.month          # 1–12
df['purchase_quarter']    = ts.dt.quarter         # 1–4
df['purchase_dayofweek']  = ts.dt.dayofweek       # 0=Mon … 6=Sun
df['purchase_hour']       = ts.dt.hour            # 0–23
df['is_weekend_purchase'] = (ts.dt.dayofweek >= 5).astype(int)  # Sat/Sun

# Hour bucket (from NB03: late-night = higher late rate)
def hour_bucket(h):
    if 0 <= h < 6:   return 'late_night'    # highest late rate
    if 6 <= h < 12:  return 'morning'
    if 12 <= h < 18: return 'afternoon'     # busiest, moderate rate
    return 'evening'                          # gradual increase toward night

df['purchase_hour_bucket'] = df['purchase_hour'].apply(hour_bucket)

print('Temporal features created:')
for col in ['purchase_year', 'purchase_month', 'purchase_quarter',
            'purchase_dayofweek', 'purchase_hour', 'is_weekend_purchase',
            'purchase_hour_bucket']:
    print(f'  {col}')

print(f'\nWeekend purchases: {df["is_weekend_purchase"].sum():,} ({df["is_weekend_purchase"].mean()*100:.1f}%)')
print(f'\nHour bucket distribution:')
print(df['purchase_hour_bucket'].value_counts().to_string())

Temporal features created:
  purchase_year
  purchase_month
  purchase_quarter
  purchase_dayofweek
  purchase_hour
  is_weekend_purchase
  purchase_hour_bucket

Weekend purchases: 22,178 (23.0%)

Hour bucket distribution:
purchase_hour_bucket
afternoon     37169
evening       33102
morning       21591
late_night     4593


---
## Step 6 — Logistics Timing Features

These features measure operational efficiency and the delivery promise window.
All are computed from timestamps **available at the time of shipment**.

| Feature | Formula | Why |
|---------|---------|-----|
| `approval_delay_h` | `approved_at – purchase_ts` in hours | Payment processing speed |
| `carrier_handling_days` | `carrier_date – approved_at` in days | Warehouse to carrier handoff speed |
| `estimated_delivery_buffer_days` | `estimated_date – purchase_ts` in days | How generous the promise window is |
| `shipping_limit_proximity_days` | `max_shipping_limit – purchase_ts` in days | Time pressure on the seller |

> ⚠️ `carrier_handling_days` uses `order_delivered_carrier_date` which is post-approval
> but **pre-delivery**. It is available before the order is delivered to the customer.
> However: it may not be available at prediction time (before shipping). Flag for NB05.

In [14]:
# ── Logistics timing features ─────────────────────────────────────────────────

# Approval delay in hours (payment processing time)
df['approval_delay_h'] = (
    df['order_approved_at'] - df['order_purchase_timestamp']
).dt.total_seconds() / 3600

# Warehouse → carrier handoff in days (operational readiness)
df['carrier_handling_days'] = (
    df['order_delivered_carrier_date'] - df['order_approved_at']
).dt.days

# Estimated delivery buffer: how long the customer was told they'd have to wait
df['estimated_delivery_buffer_days'] = (
    df['order_estimated_delivery_date'] - df['order_purchase_timestamp']
).dt.days

# Seller shipping window: time from purchase to the latest seller deadline
df['seller_shipping_window_days'] = (
    df['max_shipping_limit_date'] - df['order_purchase_timestamp']
).dt.days

# Clip extreme values (negative values = timestamp anomalies flagged in NB01)
df['approval_delay_h']              = df['approval_delay_h'].clip(lower=0)
df['carrier_handling_days']         = df['carrier_handling_days'].clip(lower=0)
df['estimated_delivery_buffer_days']= df['estimated_delivery_buffer_days'].clip(lower=1)
df['seller_shipping_window_days']   = df['seller_shipping_window_days'].clip(lower=0)

# Fill any NaN from missing timestamps
for col in ['approval_delay_h', 'carrier_handling_days',
            'estimated_delivery_buffer_days', 'seller_shipping_window_days']:
    med = df[col].median()
    n_null = df[col].isna().sum()
    df[col] = df[col].fillna(med)
    if n_null > 0:
        print(f'  {col}: filled {n_null} NaN with median ({med:.2f})')

print('Logistics timing features created.')
for col in ['approval_delay_h', 'carrier_handling_days',
            'estimated_delivery_buffer_days', 'seller_shipping_window_days']:
    print(f'  {col:<40s}: median={df[col].median():.1f}, max={df[col].max():.0f}')

Logistics timing features created.
  approval_delay_h                        : median=0.3, max=741
  carrier_handling_days                   : median=1.0, max=125
  estimated_delivery_buffer_days          : median=23.0, max=155
  seller_shipping_window_days             : median=6.0, max=1052


---
## Step 7 — Log Transformations

**EDA finding (NB03 §2):** `total_price`, `total_freight`, `total_weight_g`,
`total_volume_cm3` are all heavily right-skewed.
Log transform reduces skew and improves linear model performance.
Tree models are scale-invariant but also benefit from reduced outlier influence.

In [15]:
# ── Log(1+x) transformations ─────────────────────────────────────────────────
log_targets = [
    'total_price', 'max_item_price',
    'total_freight',
    'total_weight_g', 'max_item_weight_g',
    'total_volume_cm3', 'max_item_volume_cm3',
    'total_payment_value',
    'seller_customer_distance_km',
]

for col in log_targets:
    new_col = f'log_{col}'
    df[new_col] = np.log1p(df[col].clip(lower=0))

print('Log-transformed features created:')
for col in log_targets:
    print(f'  log_{col}')

Log-transformed features created:
  log_total_price
  log_max_item_price
  log_total_freight
  log_total_weight_g
  log_max_item_weight_g
  log_total_volume_cm3
  log_max_item_volume_cm3
  log_total_payment_value
  log_seller_customer_distance_km


---
## Step 8 — Categorical Encoding

Tree-based models can handle label encoding directly.
We encode `customer_state`, `bottleneck_seller_state`, `payment_type`,
`dominant_category_group`, and `purchase_hour_bucket`.

In [16]:
# ── Target encoding for high-cardinality categoricals ────────────────────────
# customer_state (27 states) and bottleneck_seller_state → late rate encoding
# Using full dataset here (production: use train set only — note in NB05)

global_rate = df['is_late'].mean()

def target_encode(df, col, target='is_late', smoothing=10):
    """Smooth target encoding: blend group rate with global rate."""
    stats = df.groupby(col)[target].agg(['mean', 'count']).reset_index()
    stats['encoded'] = (
        (stats['count'] * stats['mean'] + smoothing * global_rate)
        / (stats['count'] + smoothing)
    )
    return df[col].map(dict(zip(stats[col], stats['encoded']))).fillna(global_rate)


split_idx = int(len(df) * 0.80)
train_df = df.iloc[:split_idx]

def target_encode_trainonly(df_full, train_df, col, target='is_late', smoothing=10):
    stats = train_df.groupby(col)[target].agg(['mean', 'count']).reset_index()
    stats['encoded'] = (
        (stats['count'] * stats['mean'] + smoothing * global_rate)
        / (stats['count'] + smoothing)
    )
    return df_full[col].map(dict(zip(stats[col], stats['encoded']))).fillna(global_rate)

df['customer_state_te']          = target_encode_trainonly(df, train_df, 'customer_state')
df['bottleneck_seller_state_te'] = target_encode_trainonly(df, train_df, 'bottleneck_seller_state')

# ── Label encoding for low-cardinality categoricals ──────────────────────────
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in ['payment_type', 'dominant_category_group', 'purchase_hour_bucket']:
    df[f'{col}_enc'] = le.fit_transform(df[col].astype(str))

# ── Keep original states as-is for tree models (they handle strings natively) ─
# The _te and _enc versions are for linear/distance models

print('Categorical encoding done:')
print('  customer_state_te          → target encoded (smoothed)')
print('  bottleneck_seller_state_te → target encoded (smoothed)')
print('  payment_type_enc           → label encoded')
print('  dominant_category_group_enc→ label encoded')
print('  purchase_hour_bucket_enc   → label encoded')

print(f'\ncustomer_state_te stats: min={df["customer_state_te"].min():.4f}  '
      f'max={df["customer_state_te"].max():.4f}  mean={df["customer_state_te"].mean():.4f}')

Categorical encoding done:
  customer_state_te          → target encoded (smoothed)
  bottleneck_seller_state_te → target encoded (smoothed)
  payment_type_enc           → label encoded
  dominant_category_group_enc→ label encoded
  purchase_hour_bucket_enc   → label encoded

customer_state_te stats: min=0.0239  max=0.2244  mean=0.0805


---
## Step 9 — Remove Leakage Columns

These columns must be excluded from the model because they are computed
**after** the delivery event — they would not be available at prediction time.

| Column | Why it leaks |
|--------|-------------|
| `delivery_days` | Computed from actual delivery date |
| `delivery_status` | Derived from actual vs estimated delivery |
| `review_score` | Collected after delivery |
| `order_delivered_customer_date` | The outcome itself |
| All raw timestamps | Will be replaced by engineered delta features |

In [17]:
# ── Columns to drop from the model feature set ───────────────────────────────
LEAKAGE_COLS = [
    # Post-delivery outcomes
    'delivery_days',
    'delivery_status',
    'review_score',
    'order_delivered_customer_date',  # the outcome timestamp
    'carrier_handling_days',
    'seller_late_rate', # computer from mean(is_late)
    'seller_late_rate_smooth',
    'has_review'
]

# Raw timestamp columns (replaced by engineered deltas)
TIMESTAMP_COLS = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_estimated_delivery_date',
    'max_shipping_limit_date',
    'min_shipping_limit_date',
]

# ID / high-cardinality string columns (not useful as model features directly)
ID_COLS = [
    'order_id', 'customer_id', 'customer_unique_id',
    'bottleneck_seller_id',
    'customer_city',
    'dominant_category',   # replaced by dominant_category_group
]

# Geographic raw coordinates (replaced by distance feature)
GEO_COLS = [
    'customer_lat', 'customer_lng',
    'seller_lat', 'seller_lng',
    'customer_zip_code_prefix',
    'bottleneck_seller_zip',
]

all_drop = LEAKAGE_COLS + TIMESTAMP_COLS + ID_COLS + GEO_COLS

# Only drop columns that actually exist in df
cols_to_drop = [c for c in all_drop if c in df.columns]
cols_missing  = [c for c in all_drop if c not in df.columns]

print(f'Columns to drop : {len(cols_to_drop)}')
if cols_missing:
    print(f'Already absent   : {cols_missing}')

Columns to drop : 26


---
## Step 10 — Build Final Feature Matrices

We produce **two** output files:

| File | Purpose | Note |
|------|---------|------|
| `olist_features.csv` | Supervised modeling (NB05) | Includes all engineered features |
| `olist_features_cluster.csv` | Clustering (NB06) | Target-encoded features removed; uses group-level features |

In [18]:
# ── Build the modeling dataframe ──────────────────────────────────────────────
df_model = df.drop(columns=cols_to_drop)

# Also drop the raw (non-log) skewed columns — replaced by log versions
# (keep originals too for interpretability; comment out if you want fewer cols)
# df_model = df_model.drop(columns=[c for c in log_targets if c in df_model.columns])

print(f'Model dataset shape: {df_model.shape}')
print(f'\nColumn types:')
print(df_model.dtypes.value_counts().to_string())
print(f'\nRemaining NaN:')
null_counts = df_model.isnull().sum()
print(null_counts[null_counts > 0].to_string() if null_counts.any() else '  None — clean!')

Model dataset shape: (96455, 55)

Column types:
int64      24
float64    21
object      5
int32       5

Remaining NaN:
  None — clean!


In [19]:
# ── Final NaN handling before save ───────────────────────────────────────────
# Any remaining numeric NaN → median
numeric_cols = df_model.select_dtypes(include='number').columns
for col in numeric_cols:
    n = df_model[col].isna().sum()
    if n > 0:
        med = df_model[col].median()
        df_model[col] = df_model[col].fillna(med)
        print(f'  {col}: filled {n} NaN with median ({med:.4f})')

# Any remaining object NaN → 'unknown'
obj_cols = df_model.select_dtypes(include='object').columns
for col in obj_cols:
    n = df_model[col].isna().sum()
    if n > 0:
        df_model[col] = df_model[col].fillna('unknown')
        print(f'  {col}: filled {n} NaN with "unknown"')

null_after = df_model.isnull().sum().sum()
print(f'\nTotal NaN remaining: {null_after}')


Total NaN remaining: 0


In [20]:
# ── Clustering variant — remove target-encoded features ───────────────────────
# Target encoding uses is_late — must not be used in unsupervised clustering
# (it would leak the label into the cluster structure)
cluster_drop = [
    'is_late',
    'seller_late_rate',
    'seller_late_rate_smooth',
    'customer_state_te',
    'bottleneck_seller_state_te',
    'has_review',
]
cluster_drop_exist = [c for c in cluster_drop if c in df_model.columns]

df_cluster = df_model.drop(columns=cluster_drop_exist)

# Drop remaining string columns for clustering (need numeric only)
df_cluster = df_cluster.select_dtypes(include='number')

print(f'Clustering dataset shape: {df_cluster.shape}')
print(f'Model dataset shape     : {df_model.shape}')

Clustering dataset shape: (96455, 47)
Model dataset shape     : (96455, 55)


---
## Step 11 — Quality Checks

In [21]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert len(df_model) == 96455 or len(df_model) > 90000, \
    f'Unexpected row count: {len(df_model)}'
assert 'is_late' in df_model.columns, 'Target column missing!'
assert df_model['is_late'].nunique() == 2, 'Target is not binary!'
assert abs(df_model['is_late'].mean() - 0.0811) < 0.005, \
    f'Late rate changed unexpectedly: {df_model["is_late"].mean():.4f}'
assert df_model.isnull().sum().sum() == 0, 'NaN values remain!'
assert 'delivery_days' not in df_model.columns, 'Leakage column present!'
assert 'review_score' not in df_model.columns, 'Leakage column present!'
assert 'seller_customer_distance_km' in df_model.columns, 'Distance feature missing!'

print('All assertions passed ✓')
print(f'\nFinal model dataset:')
print(f'  Rows    : {len(df_model):,}')
print(f'  Columns : {df_model.shape[1]}')
print(f'  Late rate: {df_model["is_late"].mean()*100:.2f}% (must match NB01: 8.11%)')
print(f'\nFinal cluster dataset:')
print(f'  Rows    : {len(df_cluster):,}')
print(f'  Columns : {df_cluster.shape[1]}')

All assertions passed ✓

Final model dataset:
  Rows    : 96,455
  Columns : 55
  Late rate: 8.11% (must match NB01: 8.11%)

Final cluster dataset:
  Rows    : 96,455
  Columns : 47


In [22]:
# ── Feature summary ───────────────────────────────────────────────────────────
print('=== FEATURE GROUPS IN olist_features.csv ===')
print()

groups = {
    'Seller features'     : [c for c in df_model.columns if 'seller' in c.lower()],
    'Category features'   : [c for c in df_model.columns if 'cat' in c.lower()],
    'Temporal features'   : [c for c in df_model.columns if any(t in c for t in
                              ['purchase_', 'is_weekend', 'hour_bucket'])],
    'Logistics timing'    : [c for c in df_model.columns if any(t in c for t in
                              ['approval', 'carrier', 'buffer', 'window', 'spread'])],
    'Log-transformed'     : [c for c in df_model.columns if c.startswith('log_')],
    'Order complexity'    : ['n_items', 'n_unique_categories', 'n_unique_sellers',
                              'has_multi_state_sellers', 'n_unique_seller_states'],
    'Payment features'    : [c for c in df_model.columns if 'payment' in c.lower() or
                              'installment' in c.lower()],
    'Geographic'          : [c for c in df_model.columns if any(t in c for t in
                              ['state', 'distance'])],
    'Target'              : ['is_late'],
}

for grp, cols in groups.items():
    existing = [c for c in cols if c in df_model.columns]
    if existing:
        print(f'{grp} ({len(existing)}):  {existing}')
        print()

=== FEATURE GROUPS IN olist_features.csv ===

Seller features (9):  ['n_unique_sellers', 'n_unique_seller_states', 'has_multi_state_sellers', 'bottleneck_seller_state', 'seller_customer_distance_km', 'seller_order_count', 'seller_shipping_window_days', 'log_seller_customer_distance_km', 'bottleneck_seller_state_te']

Category features (10):  ['n_unique_categories', 'has_cat_bulky_home', 'has_cat_electronics', 'has_cat_fashion', 'has_cat_health_beauty', 'has_cat_media_food', 'has_cat_other', 'has_cat_sports_leisure', 'dominant_category_group', 'dominant_category_group_enc']

Temporal features (8):  ['purchase_year', 'purchase_month', 'purchase_quarter', 'purchase_dayofweek', 'purchase_hour', 'is_weekend_purchase', 'purchase_hour_bucket', 'purchase_hour_bucket_enc']

Logistics timing (4):  ['shipping_limit_spread_days', 'approval_delay_h', 'estimated_delivery_buffer_days', 'seller_shipping_window_days']

Log-transformed (9):  ['log_total_price', 'log_max_item_price', 'log_total_freight',

---
## Save Outputs

In [23]:
# ── Save ──────────────────────────────────────────────────────────────────────
import os
os.makedirs('/content/drive/MyDrive/Colab Notebooks/Olist/data/', exist_ok=True)

df_model.to_csv('/content/drive/MyDrive/Colab Notebooks/Olist/data/olist_features.csv', index=False)
df_cluster.to_csv('/content/drive/MyDrive/Colab Notebooks/Olist/data/olist_features_cluster.csv', index=False)

print('Saved successfully:')
print(f'  olist_features.csv         → {df_model.shape}   (supervised modeling — NB05)')
print(f'  olist_features_cluster.csv → {df_cluster.shape}   (clustering — NB06)')
print()
print('Column list (olist_features.csv):')
for i, col in enumerate(df_model.columns):
    print(f'  {i+1:>3}. {col}')

Saved successfully:
  olist_features.csv         → (96455, 55)   (supervised modeling — NB05)
  olist_features_cluster.csv → (96455, 47)   (clustering — NB06)

Column list (olist_features.csv):
    1. is_late
    2. has_timestamp_issue
    3. customer_state
    4. n_items
    5. total_price
    6. max_item_price
    7. total_freight
    8. total_weight_g
    9. max_item_weight_g
   10. total_volume_cm3
   11. max_item_volume_cm3
   12. n_unique_categories
   13. n_unique_sellers
   14. n_unique_seller_states
   15. shipping_limit_spread_days
   16. has_multi_state_sellers
   17. total_payment_value
   18. n_payment_methods
   19. max_installments
   20. payment_type
   21. bottleneck_seller_state
   22. seller_customer_distance_km
   23. seller_order_count
   24. has_cat_bulky_home
   25. has_cat_electronics
   26. has_cat_fashion
   27. has_cat_health_beauty
   28. has_cat_media_food
   29. has_cat_other
   30. has_cat_sports_leisure
   31. dominant_category_group
   32. purchase_year

---
## Summary

### What was built

| Output | Rows | Columns | Granularity |
|--------|------|---------|-------------|
| `olist_features.csv` | 96,455 | ~55 | One row = one order, model-ready |
| `olist_features_cluster.csv` | 96,455 | ~45 | Same orders, no target-encoded cols |

### Key engineering decisions

| Decision | Rationale |
|----------|----------|
| Bottleneck seller = `max(shipping_limit_date)` | Identifies the rate-limiting seller in multi-seller orders |
| Bayesian smoothing for seller late rate | Prevents small-sample sellers from having unstable extreme rates |
| Haversine distance (not straight-line) | More accurate for Brazil's large geographic spread |
| Category groups (7 groups, not 71 OHE) | Reduces dimensionality while preserving signal; EDA showed group-level pattern |
| Log transforms on skewed features | Reduces outlier influence; confirmed right-skew in NB03 |
| Smooth target encoding for states | Reduces noise from low-volume states |
| Two output files (model vs cluster) | Target-encoded features must not leak into clustering |

### ⚠️ Notes for NB05 (Modeling)

1. **`seller_late_rate_smooth`** — computed on full dataset. In production, compute on training set only to avoid leakage.
2. **`carrier_handling_days`** — uses carrier date which is pre-delivery but post-order. Confirm availability at prediction time.
3. **Class imbalance** — 8.1% late. Use `class_weight='balanced'` or SMOTE. Evaluate with F1 / PR-AUC.
4. **Multicollinearity** — `total_price` ↔ `total_payment_value` (near-identical). Drop one before linear models.
5. **Time-based split** recommended — train on earlier orders, test on later — to simulate real deployment.

---
**Next step:** `05_modeling.ipynb`